# ESOL Day 1：可复现的传统机器学习基线

## 实验问题

在固定的 ESOL scaffold 划分和 1024 维 ECFP 特征上，Dummy、Ridge、决策树和随机森林的训练/验证性能有何差异？

## 证据边界

- 标签使用原始 logS 空间，不使用 DeepChem 默认的 normalization transformer。
- 验证集用于比较固定候选模型；本轮不生成新的测试集分数。
- 旧 Notebook 已查看过当前 test split 的随机森林结果，因此该 test 不再视为严格未见 holdout。
- 主指标是 validation RMSE；MAE 和 R² 作为辅助指标。


In [1]:
from pathlib import Path
import json
import platform
import subprocess
import time

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

import deepchem as dc
import numpy as np
import pandas as pd
import rdkit
import scipy
import sklearn
from IPython.display import Markdown, display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

SEED = 42
RUN_ID = f"esol_ecfp_scaffold_seed{SEED}"
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Please start Jupyter from inside the ML-practice repository.")

def git_output(*args):
    try:
        return subprocess.check_output(
            ["git", *args], cwd=REPO_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "unavailable"

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / ".cache" / "deepchem"
RESULTS_DIR = REPO_ROOT / "practice" / "01_load_esol" / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_COMMIT = git_output("rev-parse", "HEAD")
BRANCH = git_output("branch", "--show-current")
GIT_DIRTY = bool(git_output("status", "--porcelain"))
VERSIONS = {
    "python": platform.python_version(),
    "deepchem": dc.__version__,
    "rdkit": rdkit.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
}

print("Repository:", REPO_ROOT.name)
print("Branch:", BRANCH)
print("Source commit:", SOURCE_COMMIT)
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))

Skipped loading some Pytorch utilities, missing a dependency. No module named 'torch'


No normalization for SPS. Feature removed!


No normalization for AvgIpc. Feature removed!


No normalization for NumAmideBonds. Feature removed!


No normalization for NumAtomStereoCenters. Feature removed!


...

Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'torch'


Skipped loading some Jax models, missing a dependency. No module named 'jax'


Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


This module requires PyTorch to be installed.


Repository: ML-practice
Branch: baseline/esol-repro
Source commit: 5bd7b5c64b07be2091c827251aeb21e23c638863
{
  "python": "3.10.20",
  "deepchem": "2.8.0",
  "rdkit": "2026.03.3",
  "numpy": "2.2.6",
  "pandas": "2.3.3",
  "scipy": "1.15.2",
  "scikit-learn": "1.7.2"
}


## 1. 显式加载数据

这里明确固定特征、划分和标签变换，避免依赖库的隐式默认值。原始数据和特征缓存写入仓库内已忽略的 `.cache/deepchem/`，不提交数据副本。


In [2]:
featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)

tasks, datasets, transformers = dc.molnet.load_delaney(
    featurizer=featurizer,
    splitter="scaffold",
    transformers=[],
    reload=True,
    data_dir=str(DATA_DIR),
    save_dir=str(DATA_DIR),
)
train_dataset, valid_dataset, test_dataset = datasets

assert transformers == [], "Day 1 requires labels in the original logS space."
print("Task:", tasks)
print("Split sizes:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("Transformers:", transformers)

Task: ['measured log solubility in mols per litre']
Split sizes: 902 113 113
Transformers: []


In [3]:
split_datasets = {
    "train": train_dataset,
    "valid": valid_dataset,
    "test": test_dataset,
}
expected_sizes = {"train": 902, "valid": 113, "test": 113}

for split_name, dataset in split_datasets.items():
    assert len(dataset) == expected_sizes[split_name]
    assert dataset.X.shape == (expected_sizes[split_name], 1024)
    assert np.isfinite(dataset.X).all()
    assert np.isfinite(dataset.y).all()
    print(
        f"{split_name:>5}: X={dataset.X.shape}, y={dataset.y.shape}, "
        f"logS=[{dataset.y.min():.3f}, {dataset.y.max():.3f}]"
    )

id_sets = {name: set(dataset.ids) for name, dataset in split_datasets.items()}
overlaps = {
    "train_valid": len(id_sets["train"] & id_sets["valid"]),
    "train_test": len(id_sets["train"] & id_sets["test"]),
    "valid_test": len(id_sets["valid"] & id_sets["test"]),
}
assert sum(overlaps.values()) == 0
print("SMILES overlap counts:", overlaps)

train: X=(902, 1024), y=(902, 1), logS=[-11.600, 1.580]
valid: X=(113, 1024), y=(113, 1), logS=[-9.332, 1.100]
 test: X=(113, 1024), y=(113, 1), logS=[-8.804, 1.070]
SMILES overlap counts: {'train_valid': 0, 'train_test': 0, 'valid_test': 0}


## 2. 固定候选模型与评价协议

本轮只比较事先固定的 5 个模型，不进行网格搜索：

1. `dummy_mean`：最简单的均值对照；
2. `ridge`：线性基线；
3. `decision_tree_overfit_diagnostic`：不限深树，仅用于展示过拟合；
4. `decision_tree_regularized`：限制深度和叶子样本数；
5. `random_forest`：Bagging 强基线。

所有模型使用同一 train/validation split，并统一记录 MAE、RMSE、R²、训练时间和参数。


In [4]:
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)

print("Train arrays:", X_train.shape, y_train.shape)
print("Validation arrays:", X_valid.shape, y_valid.shape)
print("Test predictions are intentionally not computed on Day 1.")

Train arrays: (902, 1024) (902,)
Validation arrays: (113, 1024) (113,)
Test predictions are intentionally not computed on Day 1.


In [5]:
def regression_metrics(y_true, y_pred):
    return {
        "mae_logS": float(mean_absolute_error(y_true, y_pred)),
        "rmse_logS": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [6]:
models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Ridge(alpha=1.0),
    "decision_tree_overfit_diagnostic": DecisionTreeRegressor(
        random_state=SEED
    ),
    "decision_tree_regularized": DecisionTreeRegressor(
        max_depth=5, min_samples_leaf=5, random_state=SEED
    ),
    "random_forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=SEED,
        n_jobs=-1,
    ),
}

print("Fixed candidates:", list(models))

Fixed candidates: ['dummy_mean', 'ridge', 'decision_tree_overfit_diagnostic', 'decision_tree_regularized', 'random_forest']


In [7]:
records = []
fitted_models = {}

for model_name, model in models.items():
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started
    fitted_models[model_name] = model
    params = json.dumps(model.get_params(deep=True), sort_keys=True, default=str)

    for split_name, X_split, y_split in (
        ("train", X_train, y_train),
        ("valid", X_valid, y_valid),
    ):
        prediction = model.predict(X_split)
        records.append({
            "run_id": RUN_ID,
            "model": model_name,
            "split": split_name,
            "n_samples": len(y_split),
            "seed": SEED,
            "featurizer": "ECFP radius=2 size=1024",
            "splitter": "scaffold",
            "target_transformer": "none",
            "fit_seconds": fit_seconds,
            "params": params,
            "source_commit": SOURCE_COMMIT,
            **regression_metrics(y_split, prediction),
        })

results = pd.DataFrame.from_records(records)
assert len(results) == len(models) * 2
results

,run_id,model,split,n_samples,seed,featurizer,splitter,target_transformer,fit_seconds,params,source_commit,mae_logS,rmse_logS,r2
0,esol_ecfp_scaffold_seed42,dummy_mean,train,902,42,ECFP radius=2 size=1024,scaffold,none,0.000238,"{""constant"": null, ""quantile"": null, ""strategy...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.641612,2.066724,0.000000
1,esol_ecfp_scaffold_seed42,dummy_mean,valid,113,42,ECFP radius=2 size=1024,scaffold,none,0.000238,"{""constant"": null, ""quantile"": null, ""strategy...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.635830,2.171361,-0.206488
2,esol_ecfp_scaffold_seed42,ridge,train,902,42,ECFP radius=2 size=1024,scaffold,none,0.016727,"{""alpha"": 1.0, ""copy_X"": true, ""fit_intercept""...",5bd7b5c64b07be2091c827251aeb21e23c638863,0.356172,0.529612,0.934332
3,esol_ecfp_scaffold_seed42,ridge,valid,113,42,ECFP radius=2 size=1024,scaffold,none,0.016727,"{""alpha"": 1.0, ""copy_X"": true, ""fit_intercept""...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.924364,2.379521,-0.448898
4,esol_ecfp_scaffold_seed42,decision_tree_overfit_diagnostic,train,902,42,ECFP radius=2 size=1024,scaffold,none,0.020466,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",5bd7b5c64b07be2091c827251aeb21e23c638863,0.051768,0.292828,0.979925
5,esol_ecfp_scaffold_seed42,decision_tree_overfit_diagnostic,valid,113,42,ECFP radius=2 size=1024,scaffold,none,0.020466,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.537593,1.987731,-0.011053
6,esol_ecfp_scaffold_seed42,decision_tree_regularized,train,902,42,ECFP radius=2 size=1024,scaffold,none,0.008469,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.151854,1.470852,0.493507
7,esol_ecfp_scaffold_seed42,decision_tree_regularized,valid,113,42,ECFP radius=2 size=1024,scaffold,none,0.008469,"{""ccp_alpha"": 0.0, ""criterion"": ""squared_error...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.617381,2.110553,-0.139859
8,esol_ecfp_scaffold_seed42,random_forest,train,902,42,ECFP radius=2 size=1024,scaffold,none,0.129735,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",5bd7b5c64b07be2091c827251aeb21e23c638863,0.769967,1.017978,0.757388
9,esol_ecfp_scaffold_seed42,random_forest,valid,113,42,ECFP radius=2 size=1024,scaffold,none,0.129735,"{""bootstrap"": true, ""ccp_alpha"": 0.0, ""criteri...",5bd7b5c64b07be2091c827251aeb21e23c638863,1.312411,1.703108,0.257762


In [8]:
valid_ranked = (
    results.loc[results["split"] == "valid"]
    .sort_values(["rmse_logS", "mae_logS"])
    .reset_index(drop=True)
)
best_model_name = valid_ranked.loc[0, "model"]

summary = results.pivot(
    index="model", columns="split", values=["mae_logS", "rmse_logS", "r2"]
).sort_values(("rmse_logS", "valid"))
display(summary.round(4))
print("Best model by validation RMSE:", best_model_name)
print("The test split remains unevaluated in this Day 1 run.")

mae_logS         rmse_logS              r2  \
split                               train   valid     train   valid   train   
model                                                                         
random_forest                      0.7700  1.3124    1.0180  1.7031  0.7574   
decision_tree_overfit_diagnostic   0.0518  1.5376    0.2928  1.9877  0.9799   
decision_tree_regularized          1.1519  1.6174    1.4709  2.1106  0.4935   
dummy_mean                         1.6416  1.6358    2.0667  2.1714  0.0000   
ridge                              0.3562  1.9244    0.5296  2.3795  0.9343   

                                          
split                              valid  
model                                     
random_forest                     0.2578  
decision_tree_overfit_diagnostic -0.0111  
decision_tree_regularized        -0.1399  
dummy_mean                       -0.2065  
ridge                            -0.4489

Best model by validation RMSE: random_forest
The test split remains unevaluated in this Day 1 run.


In [9]:
metrics_path = RESULTS_DIR / "baseline_metrics.csv"
config_path = RESULTS_DIR / "run_config.json"

results.to_csv(metrics_path, index=False, encoding="utf-8")
run_config = {
    "run_id": RUN_ID,
    "branch": BRANCH,
    "source_commit": SOURCE_COMMIT,
    "working_tree_dirty_during_run": GIT_DIRTY,
    "seed": SEED,
    "task": tasks[0],
    "target_space": "original logS (mols per litre)",
    "featurizer": {"name": "ECFP", "radius": 2, "size": 1024},
    "splitter": "scaffold",
    "transformers": [],
    "split_sizes": expected_sizes,
    "selection_metric": "validation RMSE",
    "best_model_by_validation_rmse": best_model_name,
    "test_policy": "Not evaluated on Day 1; the legacy notebook already exposed this split.",
    "versions": VERSIONS,
    "model_params": {name: model.get_params(deep=True) for name, model in models.items()},
}
config_path.write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)

print("Saved metrics:", metrics_path.relative_to(REPO_ROOT))
print("Saved config:", config_path.relative_to(REPO_ROOT))

Saved metrics: practice/01_load_esol/results/baseline_metrics.csv
Saved config: practice/01_load_esol/results/run_config.json


In [10]:
assert metrics_path.exists() and config_path.exists()
assert set(results["split"]) == {"train", "valid"}
assert "test" not in set(results["split"])
assert results[["mae_logS", "rmse_logS", "r2"]].notna().all().all()

dummy_valid_rmse = float(valid_ranked.loc[
    valid_ranked["model"] == "dummy_mean", "rmse_logS"
].iloc[0])
best_valid_rmse = float(valid_ranked.loc[0, "rmse_logS"])
tree_rows = results.loc[results["model"] == "decision_tree_overfit_diagnostic"]
tree_train_r2 = float(tree_rows.loc[tree_rows["split"] == "train", "r2"].iloc[0])
tree_valid_r2 = float(tree_rows.loc[tree_rows["split"] == "valid", "r2"].iloc[0])

display(Markdown(
    f"""## 3. Day 1 可验收结论

- 验证 RMSE 最低的固定候选模型：`{best_model_name}`（{best_valid_rmse:.4f} logS）。
- 它相对 Dummy 的验证 RMSE 改善：`{dummy_valid_rmse - best_valid_rmse:.4f}` logS。
- 不限深决策树的 train/valid R² 为 `{tree_train_r2:.4f}` / `{tree_valid_r2:.4f}`，训练—验证差距表明明显过拟合。
- 本轮未生成测试集预测；结论只适用于当前一次 scaffold train/validation 比较。

**Day 1 automated checks passed.**
"""
))

## 3. Day 1 可验收结论

- 验证 RMSE 最低的固定候选模型：`random_forest`（1.7031 logS）。
- 它相对 Dummy 的验证 RMSE 改善：`0.4683` logS。
- 不限深决策树的 train/valid R² 为 `0.9799` / `-0.0111`，训练—验证差距表明明显过拟合。
- 本轮未生成测试集预测；结论只适用于当前一次 scaffold train/validation 比较。

**Day 1 automated checks passed.**


## 4. 下一步

Day 2 在不改变数据划分和指标口径的前提下，增加多随机种子运行和一个树提升基线。在完成稳定性证据前，不启动 GNN 或新数据集。
